<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public abstract class Customer
{
    protected string _email;

    public int CustomerId { get; protected set; }
    public string Name { get; protected set; }
    public string Email
    {
        get => _email;
        set
        {
            if (string.IsNullOrWhiteSpace(value) || !value.Contains("@"))
                throw new ArgumentException("Некорректный email.");
            _email = value;
        }
    }

    protected Customer(int customerId, string name, string email)
    {
        if (customerId <= 0) throw new ArgumentException("ID должен быть положительным.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Имя не может быть пустым.");
        CustomerId = customerId;
        Name = name;
        Email = email;
    }

    public virtual string GetFullName() => Name;

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
        Console.WriteLine($"Email обновлён: {Email}");
    }

    public virtual void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId} | Имя: {GetFullName()} | Email: {Email}");
    }
}

public abstract class IndividualCustomer : Customer
{
    public DateTime RegistrationDate { get; protected set; }
    public int PurchaseCount { get; set; }

    protected IndividualCustomer(int customerId, string name, string email, DateTime registrationDate)
        : base(customerId, name, email)
    {
        RegistrationDate = registrationDate;
        PurchaseCount = 0;
    }

    public virtual void MakePurchase(decimal amount)
    {
        PurchaseCount++;
        Console.WriteLine($"{Name} совершил покупку на {amount:C}. Всего покупок: {PurchaseCount}");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();
        Console.WriteLine($"Регистрация: {RegistrationDate:yyyy-MM-dd} | Покупок: {PurchaseCount}");
    }
}

public interface ILoyaltyMember
{
    int LoyaltyPoints { get; }
    string Tier { get; }
    void AddLoyaltyPoints(int points);
    void RequestPrioritySupport();
}

public interface IGroupEntity
{
    string GroupName { get; }
    IReadOnlyList<Customer> Members { get; }
    void AddMember(Customer member);
}

public class VipCustomer : IndividualCustomer, ILoyaltyMember
{
    public int LoyaltyPoints { get; private set; }
    public string Tier { get; private set; }
    public bool HasDedicatedManager { get; private set; }

    public VipCustomer(int customerId, string name, string email, DateTime registrationDate, int loyaltyPoints)
        : base(customerId, name, email, registrationDate)
    {
        if (loyaltyPoints < 0) throw new ArgumentException("Баллы не могут быть отрицательными.");
        LoyaltyPoints = loyaltyPoints;
        Tier = loyaltyPoints >= 1000 ? "Platinum" : "Gold";
        HasDedicatedManager = loyaltyPoints >= 500;
    }

    public void AddLoyaltyPoints(int points)
    {
        LoyaltyPoints += points;
        Console.WriteLine($"+{points} баллов. Всего: {LoyaltyPoints}");
    }

    public void RequestPrioritySupport()
    {
        Console.WriteLine($"{Name} запрашивает приоритетную поддержку!");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();
        Console.WriteLine($"Баллы: {LoyaltyPoints} | Уровень: {Tier} | Менеджер: {(HasDedicatedManager ? "Да" : "Нет")}");
    }
}

public class RegularCustomer : IndividualCustomer
{
    public string FavoriteCategory { get; set; } = "Общее";
    public DateTime? LastEmailUpdate { get; private set; }

    public RegularCustomer(int customerId, string name, string email, DateTime registrationDate)
        : base(customerId, name, email, registrationDate)
    {
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
        Console.WriteLine($"Последнее обновление email: {LastEmailUpdate:yyyy-MM-dd HH:mm}");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();
        Console.WriteLine($"Предпочтительная категория: {FavoriteCategory}");
        if (LastEmailUpdate.HasValue)
            Console.WriteLine($"Последнее обновление email: {LastEmailUpdate:yyyy-MM-dd HH:mm}");
    }
}

public class GroupCustomer : Customer, IGroupEntity
{
    public string GroupName { get; protected set; }
    private List<Customer> _members = new List<Customer>();
    public IReadOnlyList<Customer> Members => _members.AsReadOnly();
    public DateTime CreationDate { get; private set; }
    public int MaxMembers { get; private set; } = 50;

    public GroupCustomer(int customerId, string groupName, string email)
        : base(customerId, groupName, email)
    {
        GroupName = groupName;
        CreationDate = DateTime.Now;
    }

    public override string GetFullName() => $"Группа «{GroupName}»";

    public void AddMember(Customer member)
    {
        if (member == null) throw new ArgumentNullException(nameof(member));
        if (_members.Count >= MaxMembers)
        {
            Console.WriteLine($"Группа '{GroupName}' полна.");
            return;
        }
        if (_members.Contains(member))
        {
            Console.WriteLine($"{member.Name} уже в группе.");
            return;
        }
        _members.Add(member);
        Console.WriteLine($"{member.Name} добавлен в группу '{GroupName}'");
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"Группа ID: {CustomerId} | Название: {GroupName} | Email: {Email}");
        Console.WriteLine($"Создана: {CreationDate:yyyy-MM-dd} | Участников: {_members.Count}/{MaxMembers}");
        if (_members.Any())
        {
            Console.WriteLine("Участники:");
            foreach (var m in _members)
                Console.WriteLine($"  → {m.Name} (ID: {m.CustomerId})");
        }
    }
}

var vip = new VipCustomer(1, "Алексей", "alex@vip.com", new DateTime(2022, 3, 1), 800);
var regular = new RegularCustomer(2, "Мария", "maria@test.com", new DateTime(2023, 5, 10));
var group = new GroupCustomer(100, "Команда Alpha", "alpha@org.com");

group.AddMember(vip);
group.AddMember(regular);

Customer[] customers = { vip, regular, group };
foreach (var c in customers)
{
    c.ViewProfile();
    Console.WriteLine();
}

if (vip is ILoyaltyMember vipMember)
{
    vipMember.AddLoyaltyPoints(200);
    vipMember.RequestPrioritySupport();
}

if (group is IGroupEntity groupEntity)
{
    Console.WriteLine($"Группа: {groupEntity.GroupName}, участников: {groupEntity.Members.Count}");
}

Алексей добавлен в группу 'Команда Alpha'
Мария добавлен в группу 'Команда Alpha'
ID: 1 | Имя: Алексей | Email: alex@vip.com
Регистрация: 2022-03-01 | Покупок: 0
Баллы: 800 | Уровень: Gold | Менеджер: Да

ID: 2 | Имя: Мария | Email: maria@test.com
Регистрация: 2023-05-10 | Покупок: 0
Предпочтительная категория: Общее

Группа ID: 100 | Название: Команда Alpha | Email: alpha@org.com
Создана: 2025-11-03 | Участников: 2/50
Участники:
  → Алексей (ID: 1)
  → Мария (ID: 2)

+200 баллов. Всего: 1000
Алексей запрашивает приоритетную поддержку!
Группа: Команда Alpha, участников: 2
